# Single-file Colab notebook for ISIC CLIP + FAISS + optional VLM RAG

In [ ]:
# Colab setup
!pip -q install --upgrade pip
!pip -q install torch torchvision transformers faiss-cpu scikit-learn umap-learn openai anthropic google-genai

In [ ]:
from __future__ import annotations

import base64
import json
import math
import mimetypes
import os
import zipfile
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple, Union

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer

try:
    import umap
except Exception:
    umap = None

from google.colab import files

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"}


def _require(cond: bool, msg: str) -> None:
    if not cond:
        raise ValueError(msg)


def _guess_mime_type(path: Union[str, Path]) -> str:
    mime, _ = mimetypes.guess_type(str(path))
    return mime or "image/jpeg"


def _to_absolute_path(path: Union[str, Path]) -> str:
    return str(Path(path).expanduser().resolve())


def _image_to_data_url(path: Union[str, Path]) -> str:
    path = Path(path)
    with path.open("rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{_guess_mime_type(path)};base64,{b64}"


def extract_zip(zip_path: Union[str, Path], out_dir: Union[str, Path]) -> Path:
    zip_path = Path(zip_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)
    return out_dir.resolve()


def find_image_files(root: Union[str, Path]) -> Dict[str, str]:
    root = Path(root)
    image_map: Dict[str, str] = {}
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS:
            image_map[path.stem] = str(path.resolve())
    return image_map


def show_dataset_summary(df_with_images: pd.DataFrame) -> None:
    print("=" * 80)
    print("MATCHED DATA SUMMARY")
    print("=" * 80)
    print(f"Matched rows: {len(df_with_images):,}")
    print(f"Columns: {list(df_with_images.columns)}")
    print()
    for col in ["diagnosis_1", "diagnosis_3", "sex", "anatom_site_general"]:
        if col in df_with_images.columns:
            print(f"Top values for {col}:")
            print(df_with_images[col].value_counts(dropna=False).head(10))
            print()
    if "melanocytic" in df_with_images.columns:
        print(f"Melanocytic sum: {df_with_images['melanocytic'].fillna(0).sum()}")
        print()


def load_isic_from_colab_uploads(
    upload_mode: str = "zip",
    *,
    csv_prompt: str = "Please upload the ISIC metadata CSV",
    zip_prompt: str = "Please upload a ZIP containing the image folder",
    direct_prompt: str = "Please upload all image files directly",
    images_extract_dir: str = "uploaded_isic_images",
    isic_id_col: str = "isic_id",
) -> pd.DataFrame:
    """
    Preserves the original notebook's input style:
    1. Upload CSV with files.upload()
    2. Upload either:
       - one ZIP containing the images folder, or
       - multiple image files directly
    Returns df_with_images
    """
    upload_mode = upload_mode.lower().strip()
    _require(upload_mode in {"zip", "direct"}, "upload_mode must be 'zip' or 'direct'.")

    print("=" * 80)
    print("STEP 1: CSV UPLOAD")
    print("=" * 80)
    print(csv_prompt)
    uploaded_csv = files.upload()
    _require(len(uploaded_csv) == 1, "Upload exactly one CSV file.")
    csv_filename = list(uploaded_csv.keys())[0]
    print(f"Uploaded CSV: {csv_filename}")
    df = pd.read_csv(csv_filename)
    _require(isic_id_col in df.columns, f"CSV must contain '{isic_id_col}'.")
    print(f"CSV rows: {len(df):,}")
    print()

    if upload_mode == "zip":
        print("=" * 80)
        print("STEP 2: IMAGE ZIP UPLOAD")
        print("=" * 80)
        print(zip_prompt)
        uploaded_zip = files.upload()
        _require(len(uploaded_zip) == 1, "Upload exactly one ZIP file.")
        zip_filename = list(uploaded_zip.keys())[0]
        print(f"Uploaded ZIP: {zip_filename}")

        extract_dir = extract_zip(zip_filename, images_extract_dir)
        image_map = find_image_files(extract_dir)

    else:
        print("=" * 80)
        print("STEP 2: DIRECT IMAGE UPLOAD")
        print("=" * 80)
        print(direct_prompt)
        uploaded_images = files.upload()
        _require(len(uploaded_images) > 0, "No image files were uploaded.")

        image_map = {}
        for filename in uploaded_images.keys():
            p = Path(filename)
            if p.suffix.lower() in IMAGE_EXTS:
                image_map[p.stem] = str(p.resolve())

    _require(len(image_map) > 0, "No image files were found after upload.")

    matched = df.copy()
    matched["image_path"] = matched[isic_id_col].astype(str).map(image_map)
    df_with_images = matched[matched["image_path"].notna()].copy().reset_index(drop=True)
    df_with_images["image_path"] = df_with_images["image_path"].map(_to_absolute_path)

    print("=" * 80)
    print("STEP 3: MATCHING")
    print("=" * 80)
    print(f"Images discovered: {len(image_map):,}")
    print(f"Matched rows: {len(df_with_images):,}")
    print(f"Metadata without image: {len(df) - len(df_with_images):,}")
    print(f"Image without metadata: {max(0, len(image_map) - len(df_with_images)):,}")
    print()

    show_dataset_summary(df_with_images)
    return df_with_images


class ClipEmbeddingBuilder:
    def __init__(self, model_name: str = "openai/clip-vit-base-patch32", device: Optional[str] = None):
        self.model_name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = CLIPModel.from_pretrained(model_name).to(self.device)
        self.processor = CLIPProcessor.from_pretrained(model_name)
        self.tokenizer = CLIPTokenizer.from_pretrained(model_name)
        self.model.eval()

    def encode_images(
        self,
        image_paths: Sequence[Union[str, Path]],
        *,
        batch_size: int = 16,
        normalize: bool = True,
    ) -> Tuple[np.ndarray, List[bool]]:
        all_features: List[np.ndarray] = []
        success_mask: List[bool] = []
        batch_imgs: List[Image.Image] = []

        def flush_batch() -> None:
            nonlocal batch_imgs, all_features
            if not batch_imgs:
                return
            inputs = self.processor(images=batch_imgs, return_tensors="pt", padding=True)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            with torch.no_grad():
                feats = self.model.get_image_features(**inputs)
                if normalize:
                    feats = feats / feats.norm(dim=-1, keepdim=True)
            all_features.extend(feats.cpu().numpy())
            batch_imgs = []

        for path in tqdm(list(image_paths), desc="Encoding images"):
            try:
                img = Image.open(path).convert("RGB")
                batch_imgs.append(img)
                success_mask.append(True)
                if len(batch_imgs) >= batch_size:
                    flush_batch()
            except Exception as e:
                print(f"Skipping unreadable image: {path} | {e}")
                success_mask.append(False)

        flush_batch()
        _require(len(all_features) > 0, "All image encodings failed.")

        embeddings = np.asarray(all_features, dtype=np.float32)
        return embeddings, success_mask

    def encode_text(self, text: str, normalize: bool = True) -> np.ndarray:
        inputs = self.tokenizer([text], padding=True, return_tensors="pt", truncation=True, max_length=77)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            feats = self.model.get_text_features(**inputs)
            if normalize:
                feats = feats / feats.norm(dim=-1, keepdim=True)
        return feats.cpu().numpy().astype(np.float32).reshape(-1)

    def encode_image(self, image_path: Union[str, Path], normalize: bool = True) -> np.ndarray:
        image = Image.open(image_path).convert("RGB")
        inputs = self.processor(images=image, return_tensors="pt", padding=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            feats = self.model.get_image_features(**inputs)
            if normalize:
                feats = feats / feats.norm(dim=-1, keepdim=True)
        return feats.cpu().numpy().astype(np.float32).reshape(-1)


def build_clip_artifacts(
    df_with_images: pd.DataFrame,
    *,
    out_dir: str = "clip_results",
    model_name: str = "openai/clip-vit-base-patch32",
    batch_size: int = 16,
    sample_size: Optional[int] = 500,
    random_state: int = 42,
    compute_projections: bool = True,
) -> Dict[str, object]:
    _require("image_path" in df_with_images.columns, "df_with_images must contain image_path.")
    work_df = df_with_images.copy()
    work_df["image_path"] = work_df["image_path"].map(_to_absolute_path)

    if sample_size is not None and len(work_df) > sample_size:
        work_df = work_df.sample(n=sample_size, random_state=random_state).reset_index(drop=True)
    else:
        work_df = work_df.reset_index(drop=True)

    encoder = ClipEmbeddingBuilder(model_name=model_name)
    embeddings, success_mask = encoder.encode_images(work_df["image_path"].tolist(), batch_size=batch_size)

    kept_df = work_df[success_mask].reset_index(drop=True)
    _require(len(kept_df) == len(embeddings), "Metadata and embedding counts diverged after encoding.")

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings.astype(np.float32))

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    np.save(out / "embeddings.npy", embeddings)
    kept_df.to_csv(out / "metadata.csv", index=False)
    faiss.write_index(index, str(out / "faiss_index.bin"))

    pca_2d = None
    tsne_2d = None
    umap_2d = None
    n = len(embeddings)

    if compute_projections and n >= 2:
        pca_2d = PCA(n_components=2, random_state=random_state).fit_transform(embeddings)
        np.save(out / "pca.npy", pca_2d)

        if n >= 3:
            perplexity = min(30, max(1, n - 1))
            tsne_2d = TSNE(n_components=2, random_state=random_state, perplexity=perplexity).fit_transform(embeddings)
            np.save(out / "tsne.npy", tsne_2d)

            if umap is not None:
                n_neighbors = min(15, max(2, n - 1))
                umap_2d = umap.UMAP(
                    n_components=2,
                    random_state=random_state,
                    n_neighbors=n_neighbors,
                ).fit_transform(embeddings)
                np.save(out / "umap.npy", umap_2d)

    config = {
        "clip_model": model_name,
        "device": encoder.device,
        "num_images": int(len(kept_df)),
        "embedding_dim": int(embeddings.shape[1]),
        "num_failed_images": int(len(work_df) - len(kept_df)),
        "metadata_columns": list(kept_df.columns),
        "paths_are_absolute": True,
        "sample_size_used": None if sample_size is None else int(sample_size),
    }
    with (out / "config.json").open("w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    print("=" * 80)
    print("CLIP ARTIFACTS WRITTEN")
    print("=" * 80)
    print(f"Output directory: {out.resolve()}")
    print(f"Embeddings shape: {embeddings.shape}")
    print(f"FAISS vectors: {index.ntotal}")
    print(f"Failed images: {config['num_failed_images']}")
    print()

    return {
        "embeddings": embeddings,
        "metadata": kept_df,
        "faiss_index": index,
        "config": config,
        "pca_2d": pca_2d,
        "tsne_2d": tsne_2d,
        "umap_2d": umap_2d,
    }


class BaseVLM:
    def analyze(self, context: Dict[str, object], query_image_path: Optional[Union[str, Path]] = None) -> Dict[str, object]:
        raise NotImplementedError


class OpenAIVLM(BaseVLM):
    def __init__(self, api_key: Optional[str] = None, model: str = "gpt-4o", max_output_tokens: int = 800):
        from openai import OpenAI
        self.client = OpenAI(api_key=api_key or os.getenv("OPENAI_API_KEY"))
        self.model = model
        self.max_output_tokens = max_output_tokens

    def analyze(self, context: Dict[str, object], query_image_path: Optional[Union[str, Path]] = None) -> Dict[str, object]:
        content: List[Dict[str, object]] = [{"type": "input_text", "text": build_vlm_prompt(context)}]
        image_paths: List[str] = []
        if query_image_path is not None:
            image_paths.append(str(query_image_path))
        image_paths.extend([case["image_path"] for case in context["cases"] if case.get("image_path")][:3])

        for path in image_paths:
            content.append({"type": "input_image", "image_url": _image_to_data_url(path), "detail": "high"})

        response = self.client.responses.create(
            model=self.model,
            input=[
                {
                    "role": "system",
                    "content": [
                        {
                            "type": "input_text",
                            "text": (
                                "You are assisting with educational dermatology-image retrieval analysis. "
                                "Do not state a definitive medical diagnosis."
                            ),
                        }
                    ],
                },
                {"role": "user", "content": content},
            ],
            max_output_tokens=self.max_output_tokens,
        )
        text = getattr(response, "output_text", None) or str(response)
        usage = getattr(response, "usage", None)
        total_tokens = getattr(usage, "total_tokens", None) if usage is not None else None
        return {"analysis": text, "model": self.model, "tokens_used": total_tokens}


class AnthropicVLM(BaseVLM):
    def __init__(self, api_key: Optional[str] = None, model: str = "claude-sonnet-4-6", max_tokens: int = 800):
        import anthropic
        self.client = anthropic.Anthropic(api_key=api_key or os.getenv("ANTHROPIC_API_KEY"))
        self.model = model
        self.max_tokens = max_tokens

    def analyze(self, context: Dict[str, object], query_image_path: Optional[Union[str, Path]] = None) -> Dict[str, object]:
        content: List[Dict[str, object]] = []
        image_paths: List[str] = []
        if query_image_path is not None:
            image_paths.append(str(query_image_path))
        image_paths.extend([case["image_path"] for case in context["cases"] if case.get("image_path")][:3])

        for path in image_paths:
            with open(path, "rb") as f:
                img_bytes = f.read()
            content.append(
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": _guess_mime_type(path),
                        "data": base64.b64encode(img_bytes).decode("utf-8"),
                    },
                }
            )

        content.append({"type": "text", "text": build_vlm_prompt(context)})

        response = self.client.messages.create(
            model=self.model,
            max_tokens=self.max_tokens,
            system=(
                "You are assisting with educational dermatology-image retrieval analysis. "
                "Do not state a definitive medical diagnosis."
            ),
            messages=[{"role": "user", "content": content}],
        )

        text = "\n".join(block.text for block in response.content if getattr(block, "type", None) == "text")
        usage = getattr(response, "usage", None)
        total_tokens = None
        if usage is not None:
            total_tokens = getattr(usage, "input_tokens", 0) + getattr(usage, "output_tokens", 0)
        return {"analysis": text, "model": self.model, "tokens_used": total_tokens}


class GeminiVLM(BaseVLM):
    def __init__(self, api_key: Optional[str] = None, model: str = "gemini-3-flash-preview"):
        from google import genai
        self.client = genai.Client(api_key=api_key or os.getenv("GEMINI_API_KEY"))
        self.model = model

    def analyze(self, context: Dict[str, object], query_image_path: Optional[Union[str, Path]] = None) -> Dict[str, object]:
        from google.genai import types
        contents: List[object] = []

        image_paths: List[str] = []
        if query_image_path is not None:
            image_paths.append(str(query_image_path))
        image_paths.extend([case["image_path"] for case in context["cases"] if case.get("image_path")][:3])

        for path in image_paths:
            with open(path, "rb") as f:
                contents.append(types.Part.from_bytes(data=f.read(), mime_type=_guess_mime_type(path)))

        contents.append(build_vlm_prompt(context))
        response = self.client.models.generate_content(model=self.model, contents=contents)
        return {"analysis": response.text, "model": self.model, "tokens_used": None}


class LocalEchoVLM(BaseVLM):
    def analyze(self, context: Dict[str, object], query_image_path: Optional[Union[str, Path]] = None) -> Dict[str, object]:
        cases = context["cases"][:3]
        lines = ["Local/offline summary", f"Query: {context['query']}", ""]
        for case in cases:
            lines.append(
                f"#{case['rank']} | similarity={case['similarity']:.3f} | diagnosis={case['diagnosis']} | "
                f"location={case['location']}"
            )
        lines.append("")
        lines.append("No external VLM call was made.")
        return {"analysis": "\n".join(lines), "model": "local-echo", "tokens_used": 0}


def build_vlm_prompt(context: Dict[str, object]) -> str:
    lines = [
        f"User query: {context['query']}",
        "",
        f"Retrieved {context['num_retrieved']} similar cases from the indexed dataset.",
        "Summarize recurring visual patterns, plausible differentials, uncertainty, and recommended human review points.",
        "Ground your response in the retrieved examples. Do not state a definitive diagnosis.",
        "",
        "Retrieved cases:",
    ]
    for case in context["cases"]:
        lines.extend(
            [
                f"- Case #{case['rank']} (similarity {case['similarity']:.3f})",
                f"  image_id: {case['image_id']}",
                f"  diagnosis: {case['diagnosis']}",
                f"  detailed_diagnosis: {case['detailed_diagnosis']}",
                f"  patient_age: {case['patient_age']}",
                f"  patient_sex: {case['patient_sex']}",
                f"  location: {case['location']}",
                f"  melanocytic: {case['melanocytic']}",
            ]
        )
    return "\n".join(lines)


class MedicalRAGSystem:
    def __init__(self, embeddings_dir: str = "clip_results", verbose: bool = True):
        self.embeddings_dir = Path(embeddings_dir)
        self.verbose = verbose
        self.vlm: Optional[BaseVLM] = None
        self.vlm_type: Optional[str] = None
        self._load_artifacts()
        self._load_clip_model()

    def _log(self, *args) -> None:
        if self.verbose:
            print(*args)

    def _load_artifacts(self) -> None:
        _require(self.embeddings_dir.exists(), f"Embeddings directory not found: {self.embeddings_dir}")
        self.embeddings = np.load(self.embeddings_dir / "embeddings.npy")
        self.metadata = pd.read_csv(self.embeddings_dir / "metadata.csv")
        self.faiss_index = faiss.read_index(str(self.embeddings_dir / "faiss_index.bin"))

        config_path = self.embeddings_dir / "config.json"
        self.config = json.loads(config_path.read_text()) if config_path.exists() else {}

        _require(
            len(self.metadata) == len(self.embeddings) == self.faiss_index.ntotal,
            f"Artifact mismatch: metadata={len(self.metadata)}, embeddings={len(self.embeddings)}, faiss={self.faiss_index.ntotal}",
        )
        _require("image_path" in self.metadata.columns, "metadata.csv must contain image_path.")
        self.metadata["image_path"] = self.metadata["image_path"].map(_to_absolute_path)

        self._log(f"Loaded {len(self.metadata):,} items from {self.embeddings_dir.resolve()}")

    def _load_clip_model(self) -> None:
        model_name = self.config.get("clip_model", "openai/clip-vit-base-patch32")
        self.encoder = ClipEmbeddingBuilder(model_name=str(model_name))
        self._log(f"Loaded CLIP encoder on {self.encoder.device}")

    def setup_vlm(self, vlm_type: str, api_key: Optional[str] = None, **kwargs) -> None:
        vlm_type = vlm_type.lower().strip()
        self.vlm_type = vlm_type
        if vlm_type == "openai":
            self.vlm = OpenAIVLM(api_key=api_key, **kwargs)
        elif vlm_type == "anthropic":
            self.vlm = AnthropicVLM(api_key=api_key, **kwargs)
        elif vlm_type == "gemini":
            self.vlm = GeminiVLM(api_key=api_key, **kwargs)
        elif vlm_type == "local":
            self.vlm = LocalEchoVLM()
        else:
            raise ValueError(f"Unsupported vlm_type: {vlm_type}")

    def retrieve(self, query: Union[str, Path], top_k: int = 5, query_type: str = "text") -> Dict[str, object]:
        _require(top_k > 0, "top_k must be positive.")
        top_k = min(top_k, int(self.faiss_index.ntotal))

        if query_type == "text":
            query_vec = self.encoder.encode_text(str(query))
        elif query_type == "image":
            query_vec = self.encoder.encode_image(query)
        else:
            raise ValueError("query_type must be 'text' or 'image'.")

        query_vec = query_vec.reshape(1, -1).astype(np.float32)
        distances, indices = self.faiss_index.search(query_vec, top_k)

        valid = indices[0] >= 0
        idxs = indices[0][valid]
        sims = distances[0][valid]

        retrieved = self.metadata.iloc[idxs].copy().reset_index(drop=True)
        retrieved["similarity_score"] = sims
        retrieved["rank"] = np.arange(1, len(retrieved) + 1)

        return {
            "query": str(query),
            "query_type": query_type,
            "retrieved_cases": retrieved,
            "scores": sims,
        }

    def _build_vlm_context(self, retrieval_results: Dict[str, object]) -> Dict[str, object]:
        retrieved_df = retrieval_results["retrieved_cases"]
        cases: List[Dict[str, object]] = []
        for _, row in retrieved_df.iterrows():
            cases.append(
                {
                    "rank": int(row["rank"]),
                    "similarity": float(row["similarity_score"]),
                    "image_id": row.get("isic_id", row.get("image_id", "unknown")),
                    "image_path": row.get("image_path"),
                    "diagnosis": row.get("diagnosis_1", "Unknown"),
                    "detailed_diagnosis": row.get("diagnosis_3", "Unknown"),
                    "patient_age": row.get("age_approx", "Unknown"),
                    "patient_sex": row.get("sex", "Unknown"),
                    "location": row.get("anatom_site_general", "Unknown"),
                    "melanocytic": row.get("melanocytic", "Unknown"),
                }
            )
        return {"query": retrieval_results["query"], "num_retrieved": len(cases), "cases": cases}

    def analyze_with_vlm(
        self,
        retrieval_results: Dict[str, object],
        *,
        query_image_path: Optional[Union[str, Path]] = None,
    ) -> Dict[str, object]:
        _require(self.vlm is not None, "No VLM configured. Call setup_vlm() first.")
        context = self._build_vlm_context(retrieval_results)
        return self.vlm.analyze(context, query_image_path=query_image_path)

    def complete_rag_query(
        self,
        query: Union[str, Path],
        top_k: int = 5,
        query_type: str = "text",
        use_vlm: bool = False,
        visualize: bool = False,
    ) -> Dict[str, object]:
        result = self.retrieve(query=query, top_k=top_k, query_type=query_type)
        if visualize:
            result["visualization_path"] = self.visualize_retrieval(result)
        if use_vlm:
            query_image_path = str(query) if query_type == "image" else None
            result["vlm_analysis"] = self.analyze_with_vlm(result, query_image_path=query_image_path)
        return result

    def visualize_retrieval(self, retrieval_results: Dict[str, object], save_path: str = "rag_retrieval.png", max_images: int = 6) -> str:
        retrieved_df = retrieval_results["retrieved_cases"]
        n_images = min(max_images, len(retrieved_df))
        _require(n_images > 0, "No retrieved cases to visualize.")

        cols = min(3, n_images)
        rows = math.ceil(n_images / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
        if not isinstance(axes, np.ndarray):
            axes = np.array([axes])
        axes = axes.flatten()

        for i, (_, row) in enumerate(retrieved_df.head(n_images).iterrows()):
            ax = axes[i]
            path = row.get("image_path", "")
            if path and Path(path).exists():
                ax.imshow(Image.open(path))
            else:
                ax.text(0.5, 0.5, "Image not found", ha="center", va="center")
            ax.axis("off")
            ax.set_title(
                f"#{row['rank']} | sim={row['similarity_score']:.3f}\n{row.get('isic_id', 'unknown')}\n{row.get('diagnosis_3', row.get('diagnosis_1', 'Unknown'))}",
                fontsize=10,
            )

        for j in range(i + 1, len(axes)):
            axes[j].axis("off")

        plt.suptitle(f"Retrieved images for query: {retrieval_results['query']}", fontsize=13)
        plt.tight_layout()
        save_path = str(Path(save_path).resolve())
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        return save_path

In [ ]:
# Configuration
UPLOAD_MODE = "zip"          # "zip" or "direct"
EMBEDDINGS_DIR = "clip_results"
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
BATCH_SIZE = 16
MAX_IMAGES_FOR_CLIP = 500    # set to None to use all matched images
COMPUTE_PROJECTIONS = True

# Data input using the same style as the original notebook
df_with_images = load_isic_from_colab_uploads(upload_mode=UPLOAD_MODE)

In [ ]:
# Build CLIP embeddings + FAISS index
artifacts = build_clip_artifacts(
    df_with_images,
    out_dir=EMBEDDINGS_DIR,
    model_name=CLIP_MODEL_NAME,
    batch_size=BATCH_SIZE,
    sample_size=MAX_IMAGES_FOR_CLIP,
    compute_projections=COMPUTE_PROJECTIONS,
)

# Initialize the retrieval system
rag = MedicalRAGSystem(embeddings_dir=EMBEDDINGS_DIR, verbose=True)

In [ ]:
# Example 1: text query retrieval
text_query = "malignant melanoma on the torso"
result_text = rag.complete_rag_query(
    query=text_query,
    top_k=5,
    query_type="text",
    use_vlm=False,
    visualize=True,
)

result_text["retrieved_cases"][[
    col for col in ["rank", "isic_id", "diagnosis_1", "diagnosis_3", "anatom_site_general", "similarity_score"]
    if col in result_text["retrieved_cases"].columns
]].head(5)

In [ ]:
# Example 2: image query retrieval
# Upload exactly one query image when prompted.
uploaded_query = files.upload()
query_image_path = list(uploaded_query.keys())[0]

result_image = rag.complete_rag_query(
    query=query_image_path,
    top_k=5,
    query_type="image",
    use_vlm=False,
    visualize=True,
)

result_image["retrieved_cases"][[
    col for col in ["rank", "isic_id", "diagnosis_1", "diagnosis_3", "anatom_site_general", "similarity_score"]
    if col in result_image["retrieved_cases"].columns
]].head(5)

In [ ]:
# Optional VLM step
# Choose one provider. Set the relevant API key in the environment first.
#
# import os
# os.environ["OPENAI_API_KEY"] = "..."
# os.environ["ANTHROPIC_API_KEY"] = "..."
# os.environ["GEMINI_API_KEY"] = "..."
#
# rag.setup_vlm("openai", model="gpt-4o")
# rag.setup_vlm("anthropic", model="claude-sonnet-4-6")
# rag.setup_vlm("gemini", model="gemini-3-flash-preview")
# rag.setup_vlm("local")
#
# vlm_result = rag.complete_rag_query(
#     query=text_query,
#     top_k=5,
#     query_type="text",
#     use_vlm=True,
#     visualize=False,
# )
# print(vlm_result["vlm_analysis"]["analysis"])